# NHL-Beyond-27 · Book 3 — Corsi Composites & Spicy (z) Analysis

Repo: `ewnike/NHL-Beyond-27` — *MADS Milestone I Project*
Python: 3.13.7 (pyenv env: `nhl_beyond27-3.13.7`)
Editors/Tools: VSCode, Git/GitHub, Postgres + pgAdmin
Logs: `log_utils.py` → `logs/`

**This notebook covers:**

1. Load analysis view/table
2. Define roles (Defense vs Forwards)
3. **Part A — Composite Corsi on z-scores**: unweighted vs role-weighted (D/F) → visuals → regression
4. **Part B — Spicy (within-player z)**: definition → visuals → regression
5. **Part C — Spicy-Weighted (within-player z)**: definition → visuals → regression
6. (Optional) Export small summary CSVs for the paper

**Rel-age order used throughout:** −2, −1, 0, 1, 2 (peak = 0).
We keep **composite Corsi on z-scores** (Part A) separate from **Spicy** metrics (Parts B/C).



In [10]:
import sys

import numpy as np
import pandas as pd

# Optional: these are needed later; harmless to import now
import plotly.express
import plotly.graph_objects as go
import statsmodels.api as sm
import statsmodels.formula.api as smf

print("py", sys.version.split()[0],
      "| numpy", np.__version__,
      "| pandas", pd.__version__,
      "| plotly", plotly.__version__,
      "| statsmodels", sm.__version__)


py 3.13.7 | numpy 2.3.2 | pandas 2.3.2 | plotly 6.3.0 | statsmodels 0.14.5


In [12]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf


# DB fallback loader
def load_z_table():
    """
    Try DB first (player_five_year_aligned_z), else CSV fallback at data/outputs/.
    """
    try:
        from db_utils import get_db_engine
        eng = get_db_engine()
        df = pd.read_sql("SELECT * FROM public.player_five_year_aligned_z", eng)
        print("Loaded z-table from DB:", len(df))
        return df
    except Exception as e:
        print("[Info] DB not available or table missing; using CSV fallback:", e)
        csv = Path("data/outputs/player_five_year_aligned_z.csv")
        assert csv.exists(), "CSV fallback not found: data/outputs/player_five_year_aligned_z.csv"
        df = pd.read_csv(csv)
        print("Loaded z-table from CSV:", len(df))
        return df

def load_peak_table():
    """
    EH peak season table with raw CF%/CF60/CA60 (used in Part A).
    DB optional; otherwise read data/peak_player_season_stats.csv.
    """
    try:
        from db_utils import get_db_engine
        eng = get_db_engine()
        q = ('SELECT player, season, position, "CF%", "CF/60", "CA/60" '
             'FROM public.player_peak_season')
        df = pd.read_sql(q, eng)
        print("Loaded peak table from DB:", len(df))
        return df
    except Exception as e:
        print("[Info] DB not available; using CSV fallback:", e)
        csv = Path("data/peak_player_season_stats.csv")
        assert csv.exists(), "CSV fallback not found: data/peak_player_season_stats.csv"
        df = pd.read_csv(csv)
        print("Loaded peak table from CSV:", len(df))
        return df

# --- normalize column names to lowercase, no spaces ---
def _norm_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

# >>> FIX: actually load the data <<<
df_z   = _norm_cols(load_z_table())
df_raw = _norm_cols(load_peak_table())

# If the CSV uses 'pos', make it 'position'
alias_map = {"pos": "position"}
df_z.rename(columns={k: v for k, v in alias_map.items() if k in df_z.columns}, inplace=True)
df_raw.rename(columns={k: v for k, v in alias_map.items() if k in df_raw.columns}, inplace=True)

# --- role mapping ---
def to_role(x: str) -> str:
    return "D" if str(x).upper().startswith("D") else "F"

if "position" not in df_z.columns:
    raise KeyError("Expected 'position' in z-table; got columns: " + ", ".join(df_z.columns))
if "position" not in df_raw.columns:
    raise KeyError("Expected 'position' in peak/RAW table; got columns: " + ", ".join(df_raw.columns))

df_z["role"]   = df_z["position"].map(to_role)
df_raw["role"] = df_raw["position"].map(to_role)

# enforce rel_age ordering for z table
order = [-2, -1, 0, 1, 2]
if "rel_age" in df_z.columns:
    df_z["rel_age"] = pd.Categorical(
        pd.to_numeric(df_z["rel_age"], errors="coerce"),
        categories=order, ordered=True
    )

print("Ready. (z cols):", list(df_z.columns)[:12], "…")
print("Ready. (raw cols):", list(df_raw.columns)[:12], "…")


2025-09-29 18:03:34,987 - INFO - db_utils - Using DATABASE_URL from environment.
2025-09-29 18:03:35,051 - INFO - db_utils - Using DATABASE_URL from environment.


Loaded z-table from DB: 1410
[Info] DB not available; using CSV fallback: sqlalchemy.cyextension.immutabledict.immutabledict is not a sequence
Loaded peak table from CSV: 3121
Ready. (z cols): ['player', 'position', 'peak_year', 'rel_age', 'start_year', 'season', 'age', 'cf_pct', 'cf60', 'ca60', 'cf_pct_z', 'cf60_z'] …
Ready. (raw cols): ['player', 'eh_id', 'api id', 'season', 'team', 'position', 'shoots', 'birthday', 'age', 'draft yr', 'draft rd', 'draft ov'] …


**## Part A — Composite Corsi on Z-scores (Unweighted vs Role-Weighted)**



**Goal.** Summarize play-driving with a single standardized composite, built from **within-player z-scores** (each metric centered/scaled by that player’s 5-year baseline):

* `cf_pct_z` — possession share (CF%)
* `cf60_z` — shot creation per 60 (CF/60)
* `ca60_z` — shot suppression per 60 (CA/60) *(enters with a minus sign)*

### Two composites

**1) Unweighted composite z**

$$
\text{z_comp_unweighted} ;=; \mathrm{mean}\big(cf_pct_z,; cf60_z,; -,ca60_z\big)
$$

**2) Role-weighted composite z** *(simple, fixed heuristics for this milestone)*

* **Defense (D):** emphasize suppression a bit more; creation a bit less
  $$
  \text{z_comp_weighted} ;=; 0.5,cf_pct_z ;+; 0.2,cf60_z ;-; 0.3,ca60_z
  $$

* **Forwards (F):** emphasize creation a bit more; suppression a bit less
  $$
  \text{z_comp_weighted} ;=; 0.5,cf_pct_z ;+; 0.3,cf60_z ;-; 0.2,ca60_z
  $$

> **Why z-scores?** They avoid raw-scale mixing and make the composite interpretable as “high/low **relative to the same player’s baseline**.”
> **Why these weights?** Clear, documented heuristics to keep Book 3 simple. You can tune them later with data-driven optimization or cross-validation.

### What to look for in the plots

* **By `rel_age` (−2, −1, 0, +1, +2):** trajectories around peak (0) for Defense vs Forwards.
* **Unweighted vs role-weighted:** how weighting shifts the relative separation of roles.
* **CI ribbons:** 95% confidence intervals around the mean composite by role × `rel_age`.

### Outputs in this part

* Line charts of **mean composite z** by role across `rel_age` with **95% CI** ribbons.
* Histograms / boxplots comparing the distribution of unweighted vs role-weighted composites by role.
* (Optional) Export of summary tables used to render the figures.



In [13]:
import numpy as np
import pandas as pd

dfz = df_z.copy()

# Role from position
dfz["role"] = np.where(dfz["position"].astype(str).str.upper().str.startswith("D"), "D", "F")

# Helper: mean of available values
def _mean_available(vals):
    v = [x for x in vals if pd.notna(x)]
    return float(np.mean(v)) if v else np.nan

# Unweighted composite on z-scores
def comp_unw(r):
    return _mean_available([r["cf_pct_z"], r["cf60_z"], -r["ca60_z"]])

# Role-weighted composite on z-scores
W_D = {"cf_pct_z": 0.5, "cf60_z": 0.2, "ca60_z": 0.3}
W_F = {"cf_pct_z": 0.5, "cf60_z": 0.3, "ca60_z": 0.2}

def comp_w(r):
    w = W_D if r["role"] == "D" else W_F
    num, den = 0.0, 0.0
    for k, wt in w.items():
        v = r.get(k)
        if pd.notna(v):
            if k == "ca60_z":
                v = -v  # suppress against
            num += wt * v
            den += wt
    return num / den if den else np.nan

dfz["z_comp_unweighted"] = dfz.apply(comp_unw, axis=1)
dfz["z_comp_weighted"]   = dfz.apply(comp_w,   axis=1)

# Keep rel_age ordered for plots
order = [-2, -1, 0, 1, 2]
if "rel_age" in dfz.columns:
    dfz["rel_age"] = pd.Categorical(
        pd.to_numeric(dfz["rel_age"], errors="coerce"),
        categories=order, ordered=True
    )

dfz[["player","season","role","rel_age","z_comp_unweighted","z_comp_weighted"]].head(8)


,player,season,role,rel_age,z_comp_unweighted,z_comp_weighted
0,Adam Henrique,16-17,F,-1,-0.405994,-0.428285
1,Adam Henrique,19-20,F,2,0.778992,0.931875
2,Adam Henrique,15-16,F,-2,-0.232633,-0.538525
3,Adam Henrique,17-18,F,0,0.520218,0.683384
4,Adam Henrique,18-19,F,1,-0.660584,-0.648449
5,Adam Larsson,20-21,D,0,-0.557387,-0.552449
6,Adam Larsson,22-23,D,2,1.471390,1.507333
7,Adam Larsson,21-22,D,1,0.027213,0.044183


In [14]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly import colors as pcolors


def hex_to_rgba(hex_color: str, alpha: float = 0.2) -> str:
    """Convert '#RRGGBB' to 'rgba(r,g,b,alpha)'."""
    h = hex_color.lstrip("#")
    if len(h) != 6:
        raise ValueError(f"Expected 6-digit hex like '#RRGGBB', got {hex_color}")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

def mean_ci(df: pd.DataFrame, col: str) -> pd.DataFrame:
    if not { "role", "rel_age", col }.issubset(df.columns):
        missing = { "role", "rel_age", col } - set(df.columns)
        raise KeyError(f"Column(s) missing from df: {missing}")

    g = (
        df.groupby(["role", "rel_age"])
          .agg(n=(col, "size"), mean=(col, "mean"), sd=(col, "std"))
          .reset_index()
    )
    # handle n==1 (sd is NaN) safely
    g["se"] = g["sd"].fillna(0) / np.sqrt(g["n"].clip(lower=1))
    g["lower"] = g["mean"] - 1.96 * g["se"]
    g["upper"] = g["mean"] + 1.96 * g["se"]
    return g

def ribbon_line(means: pd.DataFrame, title: str, ylab: str) -> go.Figure:
    fig = go.Figure()

    # Default palette with fallbacks for unseen roles
    base_palette = {"D": "#1f77b4", "F": "#ff7f0e"}
    fallback_cycle = pcolors.qualitative.Plotly  # a list of hex colors
    cycle_idx = 0

    # ensure rel_age is numeric for sorting; keep original for axis categories
    rel_age_order = [-2, -1, 0, 1, 2]
    means = means.copy()
    means["rel_age"] = pd.to_numeric(means["rel_age"], errors="coerce")

    # map roles to colors with stable fallback
    role_colors = {}
    for role in means["role"].dropna().unique():
        if role in base_palette:
            role_colors[role] = base_palette[role]
        else:
            role_colors[role] = fallback_cycle[cycle_idx % len(fallback_cycle)]
            cycle_idx += 1

    for role in means["role"].dropna().unique():
        sub = means.loc[means["role"] == role].sort_values("rel_age")
        sub = sub[sub["rel_age"].isin(rel_age_order)]  # keep known x
        if sub.empty:
            continue

        # use strings for categorical x so categoryarray works
        x = sub["rel_age"].astype(int).astype(str)
        c = role_colors[role]

        # Upper bound (invisible line to anchor fill)
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["upper"],
                line=dict(width=0),
                hoverinfo="skip",
                showlegend=False,
                name=f"{role} 95% CI (upper)",
            )
        )

        # Lower bound + fill between lower and the previous (upper) trace
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["lower"],
                line=dict(width=0),
                hoverinfo="skip",
                fill="tonexty",
                fillcolor=hex_to_rgba(c, alpha=0.2),
                showlegend=False,
                name=f"{role} 95% CI",
            )
        )

        # Mean line
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["mean"],
                mode="lines+markers",
                line=dict(color=c, width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    # y-range (guard against all-NaN)
    lower_min = np.nanmin(means["lower"].to_numpy()) if "lower" in means else np.nan
    upper_max = np.nanmax(means["upper"].to_numpy()) if "upper" in means else np.nan
    if np.isfinite(lower_min) and np.isfinite(upper_max):
        y_min = float(np.floor(lower_min - 0.2))
        y_max = float(np.ceil(upper_max + 0.2))
        y_range = [y_min, y_max]
    else:
        y_range = None  # let Plotly autoscale

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(categoryorder="array", categoryarray=["-2", "-1", "0", "1", "2"])
    if y_range:
        fig.update_yaxes(range=y_range, dtick=0.2)
    else:
        fig.update_yaxes(dtick=0.2)

    return fig

# ---- Example usage (make sure dfz exists and has the right columns) ----
# m_unw = mean_ci(dfz, "z_comp_unweighted")
# m_w   = mean_ci(dfz, "z_comp_weighted")
# ribbon_line(m_unw, "Composite z (Unweighted) — Mean by rel_age (95% CI)", "unweighted z").show()
# ribbon_line(m_w,   "Composite z (Role-weighted) — Mean by rel_age (95% CI)", "role-weighted z").show()


In [15]:
import plotly.express as px

# Unweighted
fig_h1 = px.histogram(
    dfz, x="z_comp_unweighted", color="role", barmode="overlay",
    nbins=40, opacity=0.6,
    title="Composite z (Unweighted) — Distribution by Role",
    labels={"z_comp_unweighted":"unweighted composite z"}
)
fig_h1.show()

# Weighted
fig_h2 = px.histogram(
    dfz, x="z_comp_weighted", color="role", barmode="overlay",
    nbins=40, opacity=0.6,
    title="Composite z (Role-weighted) — Distribution by Role",
    labels={"z_comp_weighted":"role-weighted composite z"}
)
fig_h2.show()


In [16]:
# Unweighted
fig_b1 = px.box(
    dfz, x="role", y="z_comp_unweighted",
    title="Composite z (Unweighted) — Boxplot by Role",
    labels={"z_comp_unweighted":"unweighted composite z"}
)
fig_b1.show()

# Weighted
fig_b2 = px.box(
    dfz, x="role", y="z_comp_weighted",
    title="Composite z (Role-weighted) — Boxplot by Role",
    labels={"z_comp_weighted":"role-weighted composite z"}
)
fig_b2.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go


def hex_to_rgba(hex_color: str, alpha: float = 0.2) -> str:
    """Convert '#RRGGBB' to 'rgba(r,g,b,alpha)'."""
    h = hex_color.lstrip("#")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

def mean_ci(df, col):
    g = (
        df.groupby(["role", "rel_age"], observed=True)
          .agg(n=(col, "size"), mean=(col, "mean"), sd=(col, "std"))
          .reset_index()
    )
    g["se"] = g["sd"] / np.sqrt(g["n"].clip(lower=1))
    g["lower"] = g["mean"] - 1.96 * g["se"]
    g["upper"] = g["mean"] + 1.96 * g["se"]
    return g


# def ribbon_line(means, title, ylab):
#     fig = go.Figure()
#     # palette for lines; ribbons will use rgba() version
#     palette = {"D": "#1f77b4", "F": "#ff7f0e"}

#     # ensure rel_age is properly ordered on the x-axis
#     x_order = [-2, -1, 0, 1, 2]
#     if "rel_age" in means.columns:
#         means = means.copy()
#         means["rel_age"] = (
#             means["rel_age"]
#             .astype(float)  # tolerate strings
#         )

#     for role in means["role"].dropna().unique():
#         sub = means[means["role"] == role].sort_values("rel_age")
#         x = sub["rel_age"].astype(str)

#         # Upper bound (invisible line to anchor fill)
#         fig.add_trace(
#             go.Scatter(
#                 x=x,
#                 y=sub["upper"],
#                 line=dict(width=0),
#                 hoverinfo="skip",
#                 showlegend=False,
#                 name=f"{role} 95% CI (upper)",
#             )
#         )

#         # Lower bound + fill down to it (use rgba fill color)
#         fig.add_trace(
#             go.Scatter(
#                 x=x,
#                 y=sub["lower"],
#                 line=dict(width=0),
#                 hoverinfo="skip",
#                 fill="tonexty",
#                 fillcolor=hex_to_rgba(palette.get(role, "#888888"), alpha=0.2),
#                 showlegend=False,
#                 name=f"{role} 95% CI",
#             )
#         )

#         # Mean line
#         fig.add_trace(
#             go.Scatter(
#                 x=x,
#                 y=sub["mean"],
#                 mode="lines+markers",
#                 line=dict(color=palette.get(role), width=2),
#                 marker=dict(size=7),
#                 name=f"{role} mean",
#             )
#         )

#     y_min = float(np.floor(means["lower"].min() - 0.2))
#     y_max = float(np.ceil(means["upper"].max() + 0.2))

#     fig.update_layout(
#         title=title,
#         xaxis_title="rel_age",
#         yaxis_title=ylab,
#         template="plotly_white",
#         legend=dict(title=""),
#         height=420,
#     )
#     fig.update_xaxes(categoryorder="array", categoryarray=["-2","-1","0","1","2"])
#     fig.update_yaxes(range=[y_min, y_max], dtick=0.2)

#     return fig
def ribbon_line(means, title, ylab):
    fig = go.Figure()
    palette = {"D": "#1f77b4", "F": "#ff7f0e"}

    # ensure rel_age is numeric for sorting, then use strings to respect category order
    if "rel_age" in means.columns:
        means = means.copy()
        means["rel_age"] = pd.to_numeric(means["rel_age"], errors="coerce")

    for role in means["role"].dropna().unique():
        sub = means[means["role"] == role].sort_values("rel_age")
        if sub.empty:
            continue
        x = sub["rel_age"].astype(int).astype(str)
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["mean"],
                mode="lines+markers",
                line=dict(color=palette.get(role, "#888888"), width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    # y-range based on CI (even though we don't draw the ribbon)
    lower_min = np.nanmin(means["lower"].to_numpy()) if "lower" in means else np.nan
    upper_max = np.nanmax(means["upper"].to_numpy()) if "upper" in means else np.nan
    y_range = None
    if np.isfinite(lower_min) and np.isfinite(upper_max):
        y_min = float(np.floor(lower_min - 0.2))
        y_max = float(np.ceil(upper_max + 0.2))
        y_range = [y_min, y_max]

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(categoryorder="array", categoryarray=["-2","-1","0","1","2"])
    if y_range:
        fig.update_yaxes(range=y_range, dtick=0.2)
    else:
        fig.update_yaxes(dtick=0.2)

    return fig


# Build tables and plot
m_unw = mean_ci(dfz, "z_comp_unweighted")
m_w   = mean_ci(dfz, "z_comp_weighted")

ribbon_line(m_unw, "Composite z (Unweighted) — Mean by rel_age (95% CI)", "unweighted z").show()
ribbon_line(m_w,   "Composite z (Role-weighted) — Mean by rel_age (95% CI)", "role-weighted z").show()



In [23]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go


def mean_ci(df: pd.DataFrame, col: str) -> pd.DataFrame:
    g = (
        df.groupby(["role", "rel_age"])
          .agg(n=(col, "size"), mean=(col, "mean"), sd=(col, "std"))
          .reset_index()
    )
    # sd is NaN when n==1; treat as 0 so CI collapses to mean
    g["se"] = g["sd"].fillna(0) / np.sqrt(g["n"].clip(lower=1))
    g["lower"] = g["mean"] - 1.96 * g["se"]
    g["upper"] = g["mean"] + 1.96 * g["se"]
    return g

def line_only(means: pd.DataFrame, title: str, ylab: str) -> go.Figure:
    if means.empty:
        print("[WARN] 'means' is empty.")
    means = means.copy()
    # RELIABLE x: coerce to numeric, drop NaN rel_age rows
    means["rel_age"] = pd.to_numeric(means["rel_age"], errors="coerce")
    means = means.dropna(subset=["rel_age"])

    # quick debug
    print("[DEBUG] roles:", means["role"].unique(), "rel_age:", sorted(means["rel_age"].unique()))

    palette = {"D": "#1f77b4", "F": "#ff7f0e"}
    fig = go.Figure()

    for role in means["role"].dropna().unique():
        sub = means.loc[means["role"] == role].sort_values("rel_age")
        if sub.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=sub["rel_age"],  # numeric axis → less fragile
                y=sub["mean"],
                mode="lines+markers",
                line=dict(color=palette.get(role, "#888"), width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(tickvals=[-2, -1, 0, 1, 2])
    fig.update_yaxes(dtick=0.2)
    return fig

def line_with_errorbars(means: pd.DataFrame, title: str, ylab: str) -> go.Figure:
    if means.empty:
        print("[WARN] 'means' is empty.")
    means = means.copy()
    means["rel_age"] = pd.to_numeric(means["rel_age"], errors="coerce")
    means = means.dropna(subset=["rel_age"])
    means["ci_half"] = (means["upper"] - means["mean"]).astype(float)
    means["ci_half"] = means["ci_half"].clip(lower=0).fillna(0)

    print("[DEBUG] roles:", means["role"].unique(), "rel_age:", sorted(means["rel_age"].unique()))

    palette = {"D": "#1f77b4", "F": "#ff7f0e"}
    fig = go.Figure()

    for role in means["role"].dropna().unique():
        sub = means.loc[means["role"] == role].sort_values("rel_age")
        if sub.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=sub["rel_age"],
                y=sub["mean"],
                mode="lines+markers",
                error_y=dict(type="data", array=sub["ci_half"], visible=True),
                line=dict(color=palette.get(role, "#888"), width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(tickvals=[-2, -1, 0, 1, 2])
    fig.update_yaxes(dtick=0.2)
    return fig


In [20]:
def line_only(means, title, ylab):
    fig = go.Figure()
    palette = {"D": "#1f77b4", "F": "#ff7f0e"}

    if "rel_age" in means.columns:
        means = means.copy()
        means["rel_age"] = means["rel_age"].astype(float)

    for role in means["role"].dropna().unique():
        sub = means[means["role"] == role].sort_values("rel_age")
        fig.add_trace(
            go.Scatter(
                x=sub["rel_age"].astype(str),
                y=sub["mean"],
                mode="lines+markers",
                line=dict(color=palette.get(role, "#888"), width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(categoryorder="array", categoryarray=["-2","-1","0","1","2"])
    fig.update_yaxes(dtick=0.2)
    return fig


** Quick note on negatives **
Your composites are z-scores (within-player). It’s normal for means to sit near 0 and be negative/positive depending on the season relative to the player’s baseline. That’s expected behavior, not an error.

In [ ]:
# Part A — Regression on composite z-scores (unweighted & role-weighted)
# Outcome ~ C(role) * C(rel_age, Treatment(0))
# - Intercept is Defense at rel_age=0
# - HC3 robust standard errors (heteroskedasticity-consistent)

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---- safety: keep a working copy; coerce types / categories
dfA = dfz.copy()

# role: ensure only "D" vs "F"
dfA["role"] = np.where(dfA["role"].astype(str).str.upper().str.startswith("D"), "D", "F")

# rel_age: ordered categorical with desired baseline 0
order = [-2, -1, 0, 1, 2]
dfA["rel_age"] = pd.Categorical(pd.to_numeric(dfA["rel_age"], errors="coerce"),
                                categories=order, ordered=True)

# helper to fit + tidy
def fit_and_tidy(formula: str, data: pd.DataFrame, cov_type: str = "HC3"):
    model = smf.ols(formula, data=data).fit(cov_type=cov_type)
    # tidy table
    ci = model.conf_int(alpha=0.05)
    tidy = (
        pd.DataFrame({
            "term": model.params.index,
            "coef": model.params.values,
            "se":   model.bse.values,
            "t":    model.tvalues.values,
            "p":    model.pvalues.values,
            "ci_lo": ci[0].values,
            "ci_hi": ci[1].values,
        })
        .reset_index(drop=True)
    )
    return model, tidy

# 1) Unweighted composite
form_unw = "z_comp_unweighted ~ C(role, Treatment('D')) * C(rel_age, Treatment(0))"
m_unw, tidy_unw = fit_and_tidy(form_unw, dfA, cov_type="HC3")

# 2) Role-weighted composite
form_w = "z_comp_weighted ~ C(role, Treatment('D')) * C(rel_age, Treatment(0))"
m_w, tidy_w = fit_and_tidy(form_w, dfA, cov_type="HC3")

print("Unweighted composite — OLS with HC3 SEs")
display(tidy_unw)

print("Role-weighted composite — OLS with HC3 SEs")
display(tidy_w)


## Generate adjusted means (predictions by role and rel_age + 95% CIs and plot)

In [ ]:
import itertools

import plotly.express as px


def predict_grid(model, roles=("D","F"), rels=(-2,-1,0,1,2)):
    grid = pd.DataFrame(itertools.product(roles, rels), columns=["role","rel_age"])
    # ensure rel_age matches the categorical coding (baseline at 0)
    grid["rel_age"] = pd.Categorical(grid["rel_age"], categories=order, ordered=True)
    pr = model.get_prediction(grid).summary_frame(alpha=0.05)
    out = pd.concat([grid.reset_index(drop=True), pr.reset_index(drop=True)], axis=1)
    # rename for clarity
    out = out.rename(columns={"mean":"yhat","mean_ci_lower":"ci_lo","mean_ci_upper":"ci_hi"})
    return out

pred_unw = predict_grid(m_unw)
pred_w   = predict_grid(m_w)

# Line plots with ribbons
fig_unw = px.line(
    pred_unw, x="rel_age", y="yhat", color="role", markers=True,
    category_orders={"rel_age": order},
    labels={"yhat":"Adjusted mean (unweighted z)"},
    title="Composite z (Unweighted) — Adjusted Means by rel_age (95% CI)"
)
# add ribbons
for r in pred_unw["role"].unique():
    sub = pred_unw[pred_unw["role"]==r]
    fig_unw.add_traces(px.area(sub, x="rel_age", y="ci_hi").update_traces(
        hoverinfo="skip", showlegend=False, opacity=0.15, line=dict(width=0)
    ).data)
    fig_unw.add_traces(px.area(sub, x="rel_age", y="ci_lo").update_traces(
        hoverinfo="skip", showlegend=False, opacity=0.15, line=dict(width=0)
    ).data)
fig_unw.show()

fig_w = px.line(
    pred_w, x="rel_age", y="yhat", color="role", markers=True,
    category_orders={"rel_age": order},
    labels={"yhat":"Adjusted mean (role-weighted z)"},
    title="Composite z (Role-weighted) — Adjusted Means by rel_age (95% CI)"
)
for r in pred_w["role"].unique():
    sub = pred_w[pred_w["role"]==r]
    fig_w.add_traces(px.area(sub, x="rel_age", y="ci_hi").update_traces(
        hoverinfo="skip", showlegend=False, opacity=0.15, line=dict(width=0)
    ).data)
    fig_w.add_traces(px.area(sub, x="rel_age", y="ci_lo").update_traces(
        hoverinfo="skip", showlegend=False, opacity=0.15, line=dict(width=0)
    ).data)
fig_w.show()


**Export table information to data/outputs

In [ ]:
from pathlib import Path

OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

tidy_unw.to_csv(OUT / "reg_partA_unweighted_tidy.csv", index=False)
tidy_w.to_csv(OUT / "reg_partA_weighted_tidy.csv", index=False)
pred_unw.to_csv(OUT / "reg_partA_unweighted_preds.csv", index=False)
pred_w.to_csv(OUT / "reg_partA_weighted_preds.csv", index=False)

print("Wrote:",
      OUT / "reg_partA_unweighted_tidy.csv", ",",
      OUT / "reg_partA_weighted_tidy.csv", ",",
      OUT / "reg_partA_unweighted_preds.csv", ",",
      OUT / "reg_partA_weighted_preds.csv")


## Part B — Spicy (within-player z)

**What is it?**
A composite built from within-player standardized components; it answers:  
**“Relative to this same player’s five-year baseline, how hot/cold is this season?”**

This section uses the **z-table** we built earlier (`player_five_year_aligned_z`).  
We’ll explore **spicy (unweighted)** on its own.


In [ ]:
z = df_z.copy()

# Distribution by role
fig = px.histogram(z, x="spicy_score", color="role", barmode="overlay",
                   nbins=40, opacity=0.6, title="Spicy (z) — Distribution by Role")
fig.show()

# Mean spicy by rel_age × role
if "rel_age" in z.columns:
    mean_spicy = z.groupby(["role","rel_age"], as_index=False)["spicy_score"].mean(numeric_only=True)
    fig2 = px.line(mean_spicy, x="rel_age", y="spicy_score", color="role", markers=True,
                   title="Spicy (z) — Mean by Role × rel_age")
    fig2.update_xaxes(categoryorder="array", categoryarray=[-2,-1,0,1,2])
    fig2.show()

z[["player","season","role","rel_age","spicy_score"]].head(8)


In [ ]:
# OLS: spicy ~ role * rel_age (categorical), HC3 robust SEs
if "rel_age" in z.columns:
    m_spicy = smf.ols("spicy_score ~ C(role) * C(rel_age, Treatment(0))", data=z).fit(cov_type="HC3")
else:
    m_spicy = smf.ols("spicy_score ~ C(role)", data=z).fit(cov_type="HC3")

print(m_spicy.summary())

# Tidy
coefs_spicy = pd.DataFrame({
    "term": m_spicy.params.index,
    "coef": m_spicy.params.values,
    "se":   m_spicy.bse.values,
    "p":    m_spicy.pvalues.values,
})
ci = m_spicy.conf_int()
coefs_spicy["ci_lo"] = ci[0].values
coefs_spicy["ci_hi"] = ci[1].values
coefs_spicy


## Part C — Spicy-Weighted (within-player z, role-aware)

This is the role-aware composite **in z-space** (already computed in SQL as `spicy_weighted`).  
Interpretation remains within-player: positive = hotter than that player’s own baseline, negative = colder.


In [ ]:
zw = df_z.copy()

# Scatter: spicy vs spicy-weighted, faceted by rel_age
if "rel_age" in zw.columns:
    zw["rel_age"] = pd.Categorical(pd.to_numeric(zw["rel_age"], errors="coerce"),
                                   categories=[-2,-1,0,1,2], ordered=True)

fig_sc = px.scatter(
    zw, x="spicy_score", y="spicy_weighted",
    color="role", symbol="role",
    facet_col="rel_age", facet_col_wrap=5,
    title="Spicy vs Spicy-Weighted (z) — Faceted by rel_age",
    labels={"spicy_score":"spicy", "spicy_weighted":"spicy_weighted"},
    opacity=0.6,
    category_orders={"rel_age": [-2,-1,0,1,2], "role": ["D","F"]},
)
fig_sc.show()

# Lines: mean spicy_weighted by rel_age × role
if "rel_age" in zw.columns:
    g = zw.groupby(["role","rel_age"], as_index=False)["spicy_weighted"].mean(numeric_only=True)
    fig_lines = px.line(g, x="rel_age", y="spicy_weighted", color="role", markers=True,
                        title="Spicy-Weighted (z) — Mean by Role × rel_age")
    fig_lines.update_xaxes(categoryorder="array", categoryarray=[-2,-1,0,1,2])
    fig_lines.show()

zw[["player","season","role","rel_age","spicy_weighted"]].head(8)


In [ ]:
# OLS: spicy_weighted ~ role * rel_age (categorical), HC3 robust SEs
if "rel_age" in zw.columns:
    m_sw = smf.ols("spicy_weighted ~ C(role) * C(rel_age, Treatment(0))", data=zw).fit(cov_type="HC3")
else:
    m_sw = smf.ols("spicy_weighted ~ C(role)", data=zw).fit(cov_type="HC3")

print(m_sw.summary())

# Tidy
coefs_sw = pd.DataFrame({
    "term": m_sw.params.index,
    "coef": m_sw.params.values,
    "se":   m_sw.bse.values,
    "p":    m_sw.pvalues.values,
})
ci = m_sw.conf_int()
coefs_sw["ci_lo"] = ci[0].values
coefs_sw["ci_hi"] = ci[1].values
coefs_sw


In [ ]:
OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

# Part A exports
raw[["player","season","role","cf_pct_num","cf60_num","ca60_num","corsi_weighted_role"]].to_csv(
    OUT / "role_weighted_corsi_raw.csv", index=False
)
coefs_raw.to_csv(OUT / "reg_role_weighted_corsi_coefs.csv", index=False)

# Part B exports
coefs_spicy.to_csv(OUT / "reg_spicy_coefs.csv", index=False)

# Part C exports
coefs_sw.to_csv(OUT / "reg_spicy_weighted_coefs.csv", index=False)

print("Wrote exports to:", OUT.resolve())
